# 01 — Ingest Chandrayaan-2 PDS4 Data (OHRC / TMC-2 / IIRS)

**Matches your actual Drive layout:**
```
MyDrive/26166_DATA/raw/{ohrc,tmc2,iirs}/{browse,data,geometry,miscellaneous}/{raw|calibrated}/<date>/...
```
- `data/.../*.img` (OHRC, TMC-2) or `*.qub` + `*.hdr` (IIRS) + `*.xml` PDS4 label — this is what we convert.
- `browse/` — quicklook PNGs, skipped.
- `geometry/*.csv` — per-pixel Longitude/Latitude ground-control points for OHRC & TMC-2 (used in Notebook 03, not here).
- `miscellaneous/*.lbr/.oat/.oath/.spm` — spacecraft engineering telemetry, not needed for this pipeline.

Note: **IIRS has no `geometry/` folder** in your download — it isn't georeferenced this way. That's expected for this product level; Notebook 03 handles it separately.

**Output:**
```
MyDrive/26166_DATA/processed/geotiff/{ohrc,tmc2,iirs}/<product>.tif
MyDrive/26166_DATA/processed/manifest.csv
```


## 1. Mount Drive & install geospatial libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/26166_DATA'   # <-- matches your actual folder name
RAW_ROOT   = f'{DRIVE_ROOT}/raw'
OUT_ROOT   = f'{DRIVE_ROOT}/processed/geotiff'

import os
os.makedirs(OUT_ROOT, exist_ok=True)
print('Raw root exists:', os.path.isdir(RAW_ROOT))


In [ ]:
!pip install -q rasterio pyproj tqdm
!apt-get -qq install -y gdal-bin python3-gdal > /dev/null
import rasterio
print('rasterio', rasterio.__version__)
print(rasterio._base.gdal_version())


## 2. Find every product on disk

We scan for `.xml` PDS4 labels (present for every instrument), skip anything
under a `browse/` folder, and skip `.xml` labels that belong to the
`geometry/` CSVs (those get parsed separately in Notebook 03, not here).
The recursive `rglob` already handles the nested `.../raw|calibrated/<date>/`
structure regardless of depth — no change needed for that part.


In [ ]:
from pathlib import Path

def find_pds4_labels(root):
    root = Path(root)
    labels = []
    for xml_path in root.rglob('*.xml'):
        p = str(xml_path).lower()
        if 'browse' in p or '/geometry/' in p:
            continue
        labels.append(xml_path)
    return sorted(labels)

instrument_labels = {}
for instrument in ['ohrc', 'tmc2', 'iirs']:
    found = find_pds4_labels(f'{RAW_ROOT}/{instrument}')
    instrument_labels[instrument] = found
    print(f'{instrument}: {len(found)} label(s) found')
    for p in found:
        print('   ', p)


## 3. Read pixel data — three strategies tried in order

1. **GDAL's native PDS4 driver** (`rasterio.open(xml_path)`) — works for OHRC/TMC-2 `.img` most reliably.
2. **ENVI driver** (`rasterio.open(qub_or_img_path)`) — used when a sibling `.hdr` exists, which is your IIRS case (`.qub` + `.hdr`). GDAL auto-detects ENVI format from the `.hdr` and this is often *more* reliable than PDS4 parsing for band-sequential hyperspectral cubes.
3. **Manual PDS4 label parser** — last-resort fallback if both of the above fail on a given product.


In [ ]:
import rasterio
import numpy as np

def read_via_pds4_driver(xml_path):
    try:
        with rasterio.open(str(xml_path)) as src:
            return src.read(), src.profile.copy()
    except Exception as e:
        print(f'  [PDS4 driver failed] {xml_path.name}: {e}')
        return None

def read_via_envi_driver(xml_path):
    """If a sibling .hdr exists next to the binary the label points to,
    try opening the binary directly — GDAL's ENVI driver picks up the .hdr."""
    candidates = list(xml_path.parent.glob(xml_path.stem + '.*'))
    hdr = next((c for c in candidates if c.suffix.lower() == '.hdr'), None)
    binfile = next((c for c in candidates if c.suffix.lower() in ('.qub', '.img')), None)
    if hdr is None or binfile is None:
        return None
    try:
        with rasterio.open(str(binfile)) as src:
            return src.read(), src.profile.copy()
    except Exception as e:
        print(f'  [ENVI driver failed] {binfile.name}: {e}')
        return None


### 3b. Fallback: manual PDS4 label parser (only if both drivers above fail)

In [ ]:
import xml.etree.ElementTree as ET

PDS_NS = {'pds': 'http://pds.nasa.gov/pds4/pds/v1'}

def manual_read_pds4(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    file_area = root.find('.//pds:File_Area_Observational', PDS_NS) \
             or root.find('.//pds:File_Area_Observational_Supplemental', PDS_NS)
    file_el = file_area.find('pds:File', PDS_NS)
    bin_filename = file_el.find('pds:file_name', PDS_NS).text
    bin_path = xml_path.parent / bin_filename

    array_el = (file_area.find('pds:Array_3D_Spectrum', PDS_NS)
                or file_area.find('pds:Array_3D_Image', PDS_NS)
                or file_area.find('pds:Array_2D_Image', PDS_NS))

    dtype_map = {
        'UnsignedByte': np.uint8, 'SignedByte': np.int8,
        'UnsignedMSB2': '>u2', 'UnsignedLSB2': '<u2',
        'SignedMSB2': '>i2', 'SignedLSB2': '<i2',
        'IEEE754MSBSingle': '>f4', 'IEEE754LSBSingle': '<f4',
    }
    elem_type = array_el.find('pds:Element_Array/pds:data_type', PDS_NS).text
    dtype = np.dtype(dtype_map.get(elem_type, np.uint16))

    axes = array_el.findall('pds:Axis_Array', PDS_NS)
    dims = {a.find('pds:axis_name', PDS_NS).text: int(a.find('pds:elements', PDS_NS).text)
            for a in axes}
    lines, samples, bands = dims.get('Line'), dims.get('Sample'), dims.get('Band', 1)

    data = np.fromfile(bin_path, dtype=dtype)
    data = data[: bands * lines * samples].reshape(bands, lines, samples)
    profile = {'driver': 'GTiff', 'height': lines, 'width': samples, 'count': bands, 'dtype': str(data.dtype)}
    return data, profile


## 4. Convert every found product to GeoTIFF + build a manifest

In [ ]:
import csv
from tqdm import tqdm

def read_any(xml_path, instrument):
    # IIRS: try ENVI (.hdr) first since that's the more natural format for the cube;
    # OHRC/TMC2: try PDS4 first since that's their native/only label type here.
    order = [read_via_envi_driver, read_via_pds4_driver] if instrument == 'iirs' \
            else [read_via_pds4_driver, read_via_envi_driver]
    for reader in order:
        result = reader(xml_path)
        if result is not None:
            return result
    try:
        return manual_read_pds4(xml_path)
    except Exception as e:
        print(f'  [ALL METHODS FAILED] {xml_path}: {e}')
        return None

manifest_rows = []

for instrument, labels in instrument_labels.items():
    out_dir = f'{OUT_ROOT}/{instrument}'
    os.makedirs(out_dir, exist_ok=True)

    for xml_path in tqdm(labels, desc=instrument):
        result = read_any(xml_path, instrument)
        if result is None:
            continue

        arr, profile = result
        bands, h, w = arr.shape
        out_path = os.path.join(out_dir, xml_path.stem + '.tif')

        profile.update({'height': h, 'width': w, 'count': bands, 'dtype': arr.dtype.name})
        profile.pop('crs', None)   # PDS4/ENVI raw reads rarely carry a real CRS; NB03 handles georeferencing
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(arr)

        manifest_rows.append({
            'instrument': instrument, 'product': xml_path.stem, 'geotiff_path': out_path,
            'bands': bands, 'height': h, 'width': w, 'dtype': arr.dtype.name,
            'acquisition_date_folder': xml_path.parent.name,
        })

manifest_path = f'{DRIVE_ROOT}/processed/manifest.csv'
with open(manifest_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=manifest_rows[0].keys())
    writer.writeheader()
    writer.writerows(manifest_rows)

print(f'\nConverted {len(manifest_rows)} products. Manifest saved to {manifest_path}')


## 5. Quick visual sanity check

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, instrument in zip(axes, ['ohrc', 'tmc2', 'iirs']):
    rows = [r for r in manifest_rows if r['instrument'] == instrument]
    if not rows:
        ax.set_title(f'{instrument}: none converted'); ax.axis('off'); continue
    r = rows[0]
    with rasterio.open(r['geotiff_path']) as src:
        band1 = src.read(1)
    ax.imshow(band1, cmap='gray')
    ax.set_title(f"{instrument}\n{r['product']}  {r['height']}x{r['width']}x{r['bands']}b")
    ax.axis('off')
plt.tight_layout()
plt.show()


---
### Notes
- Your OHRC product is dated **2022-09-14** and TMC-2 is dated **2019-11-25** — two
  different years/acquisitions. Before investing time in matching, visually confirm
  in Section 5 that they plausibly cover the *same ground location* — CHMAP's "layers"
  view can make nearby-but-not-identical footprints look aligned at low zoom.
- If a product fails all three read methods, print its `.xml`/`.hdr` in a scratch cell
  and check the actual tag names/values — label schemas do vary slightly by product
  version, and the manual parser's tag paths are the most likely thing to need tweaking.
- Next notebook: **02_radiometric_preprocess.ipynb**
